# report

> Structural analysis report for kosha — load-bearing nodes, communities, dependency matrix. No LLMs required.

In [ ]:
#| default_exp report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import Path, L, patch
from kosha.core import Kosha


## Utilities

In [ ]:
#| export
def _md_table(rows, cols) -> str:
	"Render rows/cols as a markdown table string."
	hdr = '| ' + ' | '.join(cols) + ' |'
	sep = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
	body = '\n'.join('| ' + ' | '.join(str(r.get(c,'')) for c in cols) + ' |' for r in rows)
	return f'{hdr}\n{sep}\n{body}'


## report_md

> Pure function: takes a `CodeGraph` and returns a Markdown string.

In [ ]:
#| export
def report_md(graph) -> str:
	"Build a Markdown structural report from a CodeGraph."
	db = graph.db
	_q = lambda sql, args=[]: list(db.q(sql, args))
	sections = []
	add = lambda title, body: sections.append(f'## {title}\n\n{body}')

	# Load-bearing nodes
	rows = _q('SELECT node, round(pagerank,5) pagerank FROM graph_nodes ORDER BY pagerank DESC LIMIT 10')
	add('Load-bearing nodes (top 10 by PageRank)', _md_table(rows, ['node','pagerank']))

	# God nodes
	rows = _q('SELECT node, in_degree FROM graph_nodes ORDER BY in_degree DESC LIMIT 10')
	add('God nodes (top 10 by in-degree)', _md_table(rows, ['node','in_degree']))

	# Deep callees
	rows = _q('SELECT node, out_degree FROM graph_nodes ORDER BY out_degree DESC LIMIT 10')
	add('Deep callees (top 10 by out-degree)', _md_table(rows, ['node','out_degree']))

	# Entry points
	rows = _q("SELECT node FROM graph_nodes WHERE in_degree=0 AND flavor='function' LIMIT 20")
	add('Entry points (in_degree=0)', '\n'.join(f'- `{r["node"]}`' for r in rows) or '_none_')

	# Leaf nodes
	rows = _q("SELECT node FROM graph_nodes WHERE out_degree=0 AND flavor='function' LIMIT 20")
	add('Leaf nodes (out_degree=0)', '\n'.join(f'- `{r["node"]}`' for r in rows) or '_none_')

	# Co-dispatch groups
	rows = _q('SELECT group_id, group_concat(node,", ") members FROM co_dispatch GROUP BY group_id LIMIT 10')
	if rows: add('Co-dispatch groups', _md_table(rows, ['group_id','members']))

	# Cross-module dependency matrix
	rows = _q("SELECT substr(caller,1,instr(caller||'.','.') - 1) caller_mod,"
	          "       substr(callee,1,instr(callee||'.','.') - 1) callee_mod,"
	          "       COUNT(*) edges FROM graph_edges"
	          " GROUP BY caller_mod, callee_mod HAVING caller_mod != callee_mod ORDER BY edges DESC LIMIT 20")
	if rows: add('Cross-module dependency matrix (top 20)', _md_table(rows, ['caller_mod','callee_mod','edges']))

	# Communities (optional)
	try:
		has_comm = _q("SELECT 1 FROM pragma_table_info('graph_nodes') WHERE name='community'")
		if has_comm:
			crows = _q('SELECT community, COUNT(*) nodes FROM graph_nodes WHERE community IS NOT NULL GROUP BY community ORDER BY nodes DESC')
			if crows: add('Communities', _md_table(crows, ['community','nodes']))
	except Exception: pass

	return '# Kosha Structural Report\n\n' + '\n\n---\n\n'.join(sections) + '\n'


## Kosha.report

In [ ]:
#| export
@patch
def report(self:Kosha, out:str='KOSHA_REPORT.md') -> Path:
	"Generate and write a structural Markdown report. Returns the output Path."
	p = Path(self.root) / out
	p.write_text(report_md(self.graph))
	return p


## Smoke test

In [ ]:
from kosha.graph import CodeGraph
g = CodeGraph(':memory:')
g.db.t.graph_nodes.insert_all([
	{'node':'a.foo','flavor':'function','file':'a.py','pagerank':0.1,'in_degree':2,'out_degree':1},
	{'node':'b.bar','flavor':'function','file':'b.py','pagerank':0.05,'in_degree':0,'out_degree':3},
], replace=True)
md = report_md(g)
assert '## Load-bearing nodes' in md
assert 'a.foo' in md
print('report_md smoke test ok')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()